# Loading, Transforming, and Delta-Loading Synthea FHIR Data

## Purpose

This project implements a complete, self-contained Databricks workflow: taking raw synthetic patient data from Synthea (in FHIR format), loading it into a Databricks workspace, parsing and flattening the nested FHIR JSON structure using PySpark, and writing the cleaned results out as Delta tables. The pipeline uses a manual-upload design — it does not depend on S3 connectivity, IAM roles, or Unity Catalog external locations — which keeps the scope focused on Databricks/Spark fundamentals rather than cloud storage integration, and makes the project runnable entirely within a Databricks Free Edition workspace.

The project demonstrates:
- Understanding the shape of FHIR data and the deliberate flattening it requires
- Loading data into a Databricks workspace without an external storage connection
- Core PySpark operations for parsing nested JSON: `explode`, schema design, `select`/`withColumn` on nested fields
- Designing and writing Delta tables from semi-structured source data
- Validating output with basic data quality checks

## Pipeline Overview
 
| Stage | Description | Output |
|---|---|---|
| 0 | Environment setup | A Databricks workspace and a small Synthea dataset |
| 1 | Download Synthea FHIR data | A folder of FHIR Bundle JSON files (one per patient) |
| 2 | Data upload to Databricks | Files available in a Unity Catalog volume |
| 3 | Raw FHIR structure exploration | Documented understanding of Bundle → entry → resource nesting |
| 4 | Resource parsing and isolation | Separate DataFrames per FHIR resource type |
| 5 | Nested field flattening | Clean, tabular DataFrames per resource type |
| 6 | Data quality pass | Deduplicated, type-cast, null-handled DataFrames |
| 7 | Delta table writes | Queryable Delta tables in the workspace catalog |
| 8 | Validation and querying | Confirmed row counts, sample queries, a join-integrity check |
| 9 | Extensions | Ideas for scaling the pipeline further |
 
---


## Stage 0: Environment Setup

A Unity Catalog volume holds the raw files:

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS learning;
CREATE SCHEMA IF NOT EXISTS learning.synthea;
CREATE VOLUME IF NOT EXISTS learning.synthea.raw_fhir;

This creates the path `/Volumes/learning/synthea/raw_fhir/`, used for upload and read access in later stages.

## Stage 1: Download Synthea FHIR Data

Download the sample FHIR R4 data from [Synthea](https://synthea.org/).

## State 2: Data Upload to Databricks

1. In the Databricks workspace, navigate to **Catalog** → `learning.synthea.raw_fhir`.
2. Use **Upload to this volume** to select the generated/downloaded `.json` files from `output/fhir/`.
3. Confirm the files landed at `/Volumes/learning/synthea/raw_fhir/`.
Verification:


In [0]:
files = dbutils.fs.ls("/Volumes/learning/synthea/raw_fhir/")
print(len(files))
display(files[:5])

100


path,name,size,modificationTime
dbfs:/Volumes/learning/synthea/raw_fhir/Adam631_Cronin387_aff8f143-2375-416f-901d-b0e4c73e3e58.json,Adam631_Cronin387_aff8f143-2375-416f-901d-b0e4c73e3e58.json,1330076,1783896005000
dbfs:/Volumes/learning/synthea/raw_fhir/Alberto639_Delatorre612_9dedae70-e46d-4de7-becd-dfb8eeaf27e6.json,Alberto639_Delatorre612_9dedae70-e46d-4de7-becd-dfb8eeaf27e6.json,580196,1783896004000
dbfs:/Volumes/learning/synthea/raw_fhir/Alene865_Lubowitz58_d4dcca82-8029-4036-a064-c5eb18291b3e.json,Alene865_Lubowitz58_d4dcca82-8029-4036-a064-c5eb18291b3e.json,1454862,1783896006000
dbfs:/Volumes/learning/synthea/raw_fhir/Alesha810_Marks830_1e0a8bd3-3b82-4f17-b1d6-19043aa0db6b.json,Alesha810_Marks830_1e0a8bd3-3b82-4f17-b1d6-19043aa0db6b.json,498300,1783896004000
dbfs:/Volumes/learning/synthea/raw_fhir/Alethea978_Crooks415_78480da7-7361-4c99-bfb9-339a403d9ae1.json,Alethea978_Crooks415_78480da7-7361-4c99-bfb9-339a403d9ae1.json,387943,1783896004000


## Stage 3: Raw FHIR Structure Exploration

Before any transformation code was written, the shape of the data was reviewed directly. Each file is a **FHIR Bundle** — a JSON object with a `type` field and an `entry` array. Each element of `entry` wraps a single **resource** (a Patient, Encounter, Condition, Observation, etc.), identified by `resource.resourceType`.

In [0]:
import json

with open("/Volumes/learning/synthea/raw_fhir/Adam631_Cronin387_aff8f143-2375-416f-901d-b0e4c73e3e58.json") as f:
    bundle = json.load(f)

print(bundle["resourceType"])  # Displays "bundle"
print(len(bundle["entry"]))  # Displays number of resources in this patient's record
print(bundle["entry"][0]["resource"]["resourceType"])  # Should display "Patient" which is usually first
print(bundle["entry"][19]["resource"]["resourceType"])  # Should display "Observation"
print(bundle["entry"][4]["resource"]["resourceType"])  # Should display "Condition"
print(bundle["entry"][3]["resource"]["resourceType"])  # Should display "Encounter"


Bundle
423
Patient
Observation
Condition
Encounter


In [0]:
print(json.dumps(bundle["entry"][0]["resource"], indent=4))  # Display patient resource
print(json.dumps(bundle["entry"][0]["resource"]["name"], indent=4))  # Display patient name object

{
    "resourceType": "Patient",
    "id": "5c818f3d-7051-4b86-8203-1dc624a91804",
    "text": {
        "status": "generated",
        "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\">Generated by <a href=\"https://github.com/synthetichealth/synthea\">Synthea</a>.Version identifier: v2.4.0-404-ge7ce2295\n .   Person seed: 7014820241964981356  Population seed: 0</div>"
    },
    "extension": [
        {
            "url": "http://hl7.org/fhir/us/core/StructureDefinition/us-core-race",
            "extension": [
                {
                    "url": "ombCategory",
                    "valueCoding": {
                        "system": "urn:oid:2.16.840.1.113883.6.238",
                        "code": "2106-3",
                        "display": "White"
                    }
                },
                {
                    "url": "text",
                    "valueString": "White"
                }
            ]
        },
        {
            "url": "http://hl7.org/fhi

In [0]:
print(bundle["entry"][19]["resource"]["subject"]["reference"]) # Display reference ID of patient
print(json.dumps(bundle["entry"][19]["resource"], indent=4))  # Display observation resource

urn:uuid:5c818f3d-7051-4b86-8203-1dc624a91804
{
    "resourceType": "Observation",
    "id": "be9e0275-9ff8-400a-8a5a-418a47926ea0",
    "status": "final",
    "category": [
        {
            "coding": [
                {
                    "system": "http://terminology.hl7.org/CodeSystem/observation-category",
                    "code": "vital-signs",
                    "display": "vital-signs"
                }
            ]
        }
    ],
    "code": {
        "coding": [
            {
                "system": "http://loinc.org",
                "code": "8302-2",
                "display": "Body Height"
            }
        ],
        "text": "Body Height"
    },
    "subject": {
        "reference": "urn:uuid:5c818f3d-7051-4b86-8203-1dc624a91804"
    },
    "encounter": {
        "reference": "urn:uuid:fae3b828-72f9-425d-910e-b97ce1d2431e"
    },
    "effectiveDateTime": "2010-01-15T07:17:50-05:00",
    "issued": "2010-01-15T07:17:50.590-05:00",
    "valueQuantity": {
  

In [0]:
print(json.dumps(bundle["entry"][4]["resource"], indent=4))  # Display condition resource

{
    "resourceType": "Condition",
    "id": "00ae9218-0a21-4ac9-9b21-c3a30a527f5a",
    "clinicalStatus": {
        "coding": [
            {
                "system": "http://terminology.hl7.org/CodeSystem/condition-clinical",
                "code": "resolved"
            }
        ]
    },
    "verificationStatus": {
        "coding": [
            {
                "system": "http://terminology.hl7.org/CodeSystem/condition-ver-status",
                "code": "confirmed"
            }
        ]
    },
    "code": {
        "coding": [
            {
                "system": "http://snomed.info/sct",
                "code": "232353008",
                "display": "Perennial allergic rhinitis with seasonal variation"
            }
        ],
        "text": "Perennial allergic rhinitis with seasonal variation"
    },
    "subject": {
        "reference": "urn:uuid:5c818f3d-7051-4b86-8203-1dc624a91804"
    },
    "encounter": {
        "reference": "urn:uuid:4b6690bd-5672-407b-a7ee-5

In [0]:
print(json.dumps(bundle["entry"][3]["resource"], indent=4))  # Display encounter resource

{
    "resourceType": "Encounter",
    "id": "4b6690bd-5672-407b-a7ee-5dcd3becaf60",
    "status": "finished",
    "class": {
        "system": "http://terminology.hl7.org/CodeSystem/v3-ActCode",
        "code": "AMB"
    },
    "type": [
        {
            "coding": [
                {
                    "system": "http://snomed.info/sct",
                    "code": "185345009",
                    "display": "Encounter for symptom"
                }
            ],
            "text": "Encounter for symptom"
        }
    ],
    "subject": {
        "reference": "urn:uuid:5c818f3d-7051-4b86-8203-1dc624a91804",
        "display": "Mr. Adam631 Cronin387"
    },
    "participant": [
        {
            "individual": {
                "reference": "urn:uuid:0000016d-3a85-4cca-0000-0000000000aa",
                "display": "Dr. Jacquelyn628 Pouros728"
            }
        }
    ],
    "period": {
        "start": "2000-01-02T07:17:50-05:00",
        "end": "2000-01-16T07:17:50-05:0

Reviewing full `Patient` and `Observation` resources surfaced three structural characteristics that shaped the rest of the pipeline:
- Fields are deeply nested (`name` is a list of objects, each with `given` as a list of strings)
- Cross-references between resources use `reference` strings like `"urn:uuid:abc-123"`, not simple foreign keys
- Resource shapes vary by type, requiring type-specific parsing logic
This exploration step proved essential — several downstream schema issues (see Stage 4) trace directly back to structural quirks that were only visible by inspecting real records rather than assuming a shape.


## Stage 4: Resource Parsing and Isolation

All files were loaded into Spark and separated by resource type.
 
An initial pass relied on Spark's default schema inference:
 
```python
raw_df = spark.read.option("multiline", "true") \
    .json("/Volumes/learning/synthea/raw_fhir/*.json")
 
raw_df.printSchema()
```
 
**Schema inference issue encountered:** Spark infers a single unified schema across every resource type in a Bundle. Because field names like `name` and `code` are reused across different resource types with different underlying shapes (e.g., `Patient.name` is an array of structs, while `Organization.name` is a plain string), inference collapsed some fields to the wrong type, surfacing as `INVALID_EXTRACT_BASE_FIELD_TYPE` errors when flattening logic expected a struct/array but received a string.
 
**Resolution:** an explicit schema was defined and supplied to the reader up front, rather than relying on inference. This prevented the type collapse entirely and removed the need for any inference-based workaround:


In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, DoubleType
)

# Reusable FHIR sub-shapes
reference_type = StructType([StructField("reference", StringType())])
coding_type = StructType([
    StructField("system", StringType()),
    StructField("code", StringType()),
    StructField("display", StringType())
])
codeable_concept_type = StructType([
    StructField("coding", ArrayType(coding_type)),
    StructField("text", StringType())
])
period_type = StructType([
    StructField("start", StringType()),
    StructField("end", StringType())
])
human_name_type = StructType([
    StructField("family", StringType()),
    StructField("given", ArrayType(StringType()))
])
address_type = StructType([
    StructField("city", StringType()),
    StructField("state", StringType())
])

# One combined resource schema covering every field you use across
# Patient, Encounter, Condition, and Observation
resource_schema = StructType([
    StructField("resourceType", StringType()),
    StructField("id", StringType()),
    StructField("gender", StringType()),
    StructField("birthDate", StringType()),
    StructField("name", ArrayType(human_name_type)),
    StructField("address", ArrayType(address_type)),
    StructField("status", StringType()),
    StructField("type", ArrayType(codeable_concept_type)),
    StructField("subject", reference_type),
    StructField("period", period_type),
    StructField("code", codeable_concept_type),
    StructField("onsetDateTime", StringType()),
    StructField("clinicalStatus", codeable_concept_type),
    StructField("encounter", reference_type),
    StructField("effectiveDateTime", StringType()),
    StructField("valueQuantity", StructType([
        StructField("value", DoubleType()),
        StructField("unit", StringType())
    ])),
    StructField("valueString", StringType()),
    StructField("valueCodeableConcept", codeable_concept_type)
])

bundle_schema = StructType([
    StructField("entry", ArrayType(
        StructType([StructField("resource", resource_schema)])
    ))
])

In [0]:
raw_df = spark.read.schema(bundle_schema).option("multiline", "true").json("/Volumes/learning/synthea/raw_fhir/*.json")

Each row of `raw_df` is one whole Bundle. The `entry` array was exploded to work with individual resources:

In [0]:
from pyspark.sql.functions import explode, col

entries_df = raw_df.select(explode(col("entry")).alias("entry"))
entries_df = entries_df.select(
    col("entry.resource.resourceType").alias("resourceType"),
    col("entry.resource").alias("resource")
)

entries_df.groupBy("resourceType").count().orderBy(col("count").desc()).show()

+--------------------+-----+
|        resourceType|count|
+--------------------+-----+
|         Observation|22080|
|               Claim| 5523|
|           Encounter| 4430|
|ExplanationOfBenefit| 4430|
|           Procedure| 3628|
|    DiagnosticReport| 1594|
|        Immunization| 1218|
|   MedicationRequest| 1093|
|           Condition|  799|
|            CarePlan|  350|
|            CareTeam|  350|
|                Goal|  276|
|        Organization|  255|
|        Practitioner|  255|
|             Patient|  100|
|  AllergyIntolerance|   41|
|        ImagingStudy|   25|
|              Device|    6|
|MedicationAdminis...|    6|
+--------------------+-----+



The resulting resource-type counts (Patient, Encounter, Condition, Observation, and others, with counts scaling as expected, far more Observations than Patients) confirmed the schema and parsing logic were correct before any flattening began.
 
Four resource types were targeted for this project: **Patient, Encounter, Condition, Observation.**


In [0]:
patient_raw = entries_df.filter(col("resourceType") == "Patient").select("resource.*")
encounter_raw = entries_df.filter(col("resourceType") == "Encounter").select("resource.*")
condition_raw = entries_df.filter(col("resourceType") == "Condition").select("resource.*")
observation_raw = entries_df.filter(col("resourceType") == "Observation").select("resource.*")

## Stage 5: Nested Field Flattening

Each resource type required its own flattening logic, since FHIR nests fields differently per type.

In [0]:
from pyspark.sql.functions import col, expr, to_json, from_json

patient_clean = patient_raw.select(
    col("id").alias("patient_id"),
    col("gender"),
    col("birthDate").alias("birth_date"),
    expr("name[0].family").alias("last_name"),
    expr("name[0].given[0]").alias("first_name"),
    expr("address[0].city").alias("city"),
    expr("address[0].state").alias("state")
)

patient_clean.show(5)

+--------------------+------+----------+-------------+----------+---------+-------------+
|          patient_id|gender|birth_date|    last_name|first_name|     city|        state|
+--------------------+------+----------+-------------+----------+---------+-------------+
|b0f49c80-b59b-4df...|female|1927-06-20|Echevarría842|  Ester635|Westfield|Massachusetts|
|973570aa-ab35-45a...|female|1940-03-02|    Zelaya592| Susana117|     Lynn|Massachusetts|
|11261f38-18f9-409...|  male|1932-12-18|     Moore224|Patrick786|   Boston|Massachusetts|
|980f0211-51a3-484...|female|1939-04-03|  Dietrich576|Phillis443|Pepperell|Massachusetts|
|a2bc2feb-d766-47d...|female|1994-02-26|     Salas880|Beatriz277|   Auburn|Massachusetts|
+--------------------+------+----------+-------------+----------+---------+-------------+
only showing top 5 rows


In [0]:
encounter_clean = encounter_raw.select(
    col("id").alias("encounter_id"),
    expr("split(subject.reference, ':')[2]").alias("patient_id"),
    col("status"),
    expr("type[0].coding[0].display").alias("encounter_type"),
    col("period.start").alias("start_time"),
    col("period.end").alias("end_time")
)

encounter_clean.show(5)

+--------------------+--------------------+--------+--------------------+--------------------+--------------------+
|        encounter_id|          patient_id|  status|      encounter_type|          start_time|            end_time|
+--------------------+--------------------+--------+--------------------+--------------------+--------------------+
|064756d7-d524-44f...|b0f49c80-b59b-4df...|finished|Encounter for sym...|1961-08-30T10:16:...|1961-09-04T10:16:...|
|cd1a9a2c-9908-4ff...|b0f49c80-b59b-4df...|finished|Encounter for sym...|1963-07-04T10:16:...|1963-07-04T10:31:...|
|de4b6207-187b-458...|b0f49c80-b59b-4df...|finished|General examinati...|1967-06-26T10:16:...|1967-06-26T10:46:...|
|c642773d-c899-440...|b0f49c80-b59b-4df...|finished|Encounter for pro...|1971-06-09T10:16:...|1971-06-09T10:31:...|
|827a7f7f-27c0-423...|b0f49c80-b59b-4df...|finished|Encounter for sym...|1977-10-17T10:16:...|1977-10-17T10:31:...|
+--------------------+--------------------+--------+--------------------

In [0]:
condition_clean = condition_raw.select(
    col("id").alias("condition_id"),
    expr("split(subject.reference, ':')[2]").alias("patient_id"),
    expr("code.coding[0].code").alias("condition_code"),
    expr("code.coding[0].display").alias("condition_description"),
    col("onsetDateTime").alias("onset_date"),
    expr("clinicalStatus.coding[0].code").alias("clinical_status")
)

condition_clean.show(5)

+--------------------+--------------------+--------------+---------------------+--------------------+---------------+
|        condition_id|          patient_id|condition_code|condition_description|          onset_date|clinical_status|
+--------------------+--------------------+--------------+---------------------+--------------------+---------------+
|b6a1f7e2-7905-493...|b0f49c80-b59b-4df...|      44054006|             Diabetes|1961-09-04T10:16:...|         active|
|357225ec-3e14-4ae...|b0f49c80-b59b-4df...|     271737000|    Anemia (disorder)|1961-09-04T10:16:...|         active|
|3ced7611-8fd0-4a9...|b0f49c80-b59b-4df...|     302870006| Hypertriglyceride...|1963-07-08T10:16:...|         active|
|3091377d-50d7-490...|b0f49c80-b59b-4df...|     237602007| Metabolic syndrom...|1967-06-26T10:16:...|         active|
|2f90633c-9082-457...|b0f49c80-b59b-4df...|     422034002| Diabetic retinopa...|1967-06-26T10:16:...|         active|
+--------------------+--------------------+-------------

**Observation:** FHIR stores an Observation's result in one of several possible fields depending on the value's data type (`valueQuantity` for numeric results, `valueString` for text, `valueCodeableConcept` for coded results), with only one populated per record. `coalesce` was used to collapse these into a single `value` column by taking the first non-null field, in order, for each row:

In [0]:
from pyspark.sql.functions import coalesce

observation_clean = observation_raw.select(
    col("id").alias("observation_id"),
    expr("split(subject.reference, ':')[2]").alias("patient_id"),
    expr("split(encounter.reference, ':')[2]").alias("encounter_id"),
    expr("code.coding[0].display").alias("observation_name"),
    col("effectiveDateTime").alias("observed_at"),
    coalesce(
        col("valueQuantity.value").cast("string"),
        col("valueString"),
        expr("valueCodeableConcept.coding[0].display")
    ).alias("value"),
    col("valueQuantity.unit").alias("unit")
)

observation_clean.show(5)

+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+-------+
|      observation_id|          patient_id|        encounter_id|    observation_name|         observed_at|             value|   unit|
+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+-------+
|54005564-d083-4d0...|b0f49c80-b59b-4df...|72ec4e69-66f5-4cf...|         Body Height|2009-12-21T09:16:...|161.60744331192961|     cm|
|bb3128bb-054a-47a...|b0f49c80-b59b-4df...|72ec4e69-66f5-4cf...|Pain severity - 0...|2009-12-21T09:16:...|0.1598744891897237|{score}|
|24255295-d696-40c...|b0f49c80-b59b-4df...|72ec4e69-66f5-4cf...|         Body Weight|2009-12-21T09:16:...| 72.88768760637842|     kg|
|31e35602-d971-447...|b0f49c80-b59b-4df...|72ec4e69-66f5-4cf...|     Body Mass Index|2009-12-21T09:16:...|27.908175991515364|  kg/m2|
|0041f10f-3f29-465...|b0f49c80-b59b-4df...|72ec4e69-66f5-4cf..

**Scope note:** real FHIR data includes considerably more fields and edge cases than captured here (multiple addresses, historical names, missing values, resource versioning, etc.). This project intentionally captures a focused subset of fields relevant to the four target resource types, rather than a full FHIR implementation.

## Stage 6: Data Quality Pass

Basic checks and cleanup were run on each cleaned DataFrame before writing to Delta:

In [0]:
# Check for unexpected nulls in key fields
patient_clean.filter(col("patient_id").isNull()).count()
observation_clean.filter(col("observation_id").isNull()).count()
condition_clean.filter(col("condition_id").isNull()).count()
encounter_clean.filter(col("encounter_id").isNull()).count()

# Check for duplicate IDs
from pyspark.sql.functions import count as spark_count
patient_clean.groupBy("patient_id").count().filter(col("count") > 1).count()  
observation_clean.groupBy("observation_id").count().filter(col("count") > 1).count()  
condition_clean.groupBy("condition_id").count().filter(col("count") > 1).count()  
encounter_clean.groupBy("encounter_id").count().filter(col("count") > 1).count()  

0

In [0]:
from pyspark.sql.functions import to_date, to_timestamp
patient_clean = patient_clean.withColumn("birth_date", to_date("birth_date"))

patient_clean.show(5)

+--------------------+------+----------+-------------+----------+---------+-------------+
|          patient_id|gender|birth_date|    last_name|first_name|     city|        state|
+--------------------+------+----------+-------------+----------+---------+-------------+
|b0f49c80-b59b-4df...|female|1927-06-20|Echevarría842|  Ester635|Westfield|Massachusetts|
|973570aa-ab35-45a...|female|1940-03-02|    Zelaya592| Susana117|     Lynn|Massachusetts|
|11261f38-18f9-409...|  male|1932-12-18|     Moore224|Patrick786|   Boston|Massachusetts|
|980f0211-51a3-484...|female|1939-04-03|  Dietrich576|Phillis443|Pepperell|Massachusetts|
|a2bc2feb-d766-47d...|female|1994-02-26|     Salas880|Beatriz277|   Auburn|Massachusetts|
+--------------------+------+----------+-------------+----------+---------+-------------+
only showing top 5 rows


In [0]:
encounter_clean = encounter_clean.withColumn("start_time", to_timestamp("start_time")).withColumn("end_time", to_timestamp("end_time"))

encounter_clean.show(5)

+--------------------+--------------------+--------+--------------------+-------------------+-------------------+
|        encounter_id|          patient_id|  status|      encounter_type|         start_time|           end_time|
+--------------------+--------------------+--------+--------------------+-------------------+-------------------+
|064756d7-d524-44f...|b0f49c80-b59b-4df...|finished|Encounter for sym...|1961-08-30 14:16:50|1961-09-04 14:16:50|
|cd1a9a2c-9908-4ff...|b0f49c80-b59b-4df...|finished|Encounter for sym...|1963-07-04 14:16:50|1963-07-04 14:31:50|
|de4b6207-187b-458...|b0f49c80-b59b-4df...|finished|General examinati...|1967-06-26 14:16:50|1967-06-26 14:46:50|
|c642773d-c899-440...|b0f49c80-b59b-4df...|finished|Encounter for pro...|1971-06-09 14:16:50|1971-06-09 14:31:50|
|827a7f7f-27c0-423...|b0f49c80-b59b-4df...|finished|Encounter for sym...|1977-10-17 14:16:50|1977-10-17 14:31:50|
+--------------------+--------------------+--------+--------------------+---------------

## Stage 7: Delta Table Writes

Each cleaned DataFrame was written as a managed Delta table:

In [0]:
patient_clean.write.mode("overwrite").saveAsTable("learning.synthea.patients")
encounter_clean.write.mode("overwrite").saveAsTable("learning.synthea.encounters")
condition_clean.write.mode("overwrite").saveAsTable("learning.synthea.conditions")
observation_clean.write.mode("overwrite").saveAsTable("learning.synthea.observations")

`mode("overwrite")` was used throughout pipeline iteration. A future incremental-load version would switch to `mode("append")` paired with a `MERGE INTO` upsert strategy to avoid re-inserting unchanged records on each run (see Stage 9).

## Stage 8: Validation and Querying

Table integrity was confirmed with row counts and a referential check:

In [0]:
%sql
SELECT COUNT(*) FROM learning.synthea.patients;
SELECT COUNT(*) FROM learning.synthea.observations;

-- Sanity check: every encounter should reference a real patient; should return null
SELECT COUNT(*) FROM learning.synthea.encounters e
LEFT JOIN learning.synthea.patients p ON e.patient_id = p.patient_id
WHERE p.patient_id IS NULL;

COUNT(*)
0


The referential check returns 0, confirming the `reference` parsing logic in Stage 5 produced valid, joinable keys across tables.
 
An analytical query confirmed the tables function together as intended:


In [0]:
%sql
SELECT p.gender, c.condition_description, COUNT(*) AS condition_count
FROM learning.synthea.conditions c
JOIN learning.synthea.patients p ON c.patient_id = p.patient_id
GROUP BY p.gender, c.condition_description
ORDER BY condition_count DESC
LIMIT 20;

gender,condition_description,condition_count
female,Normal pregnancy,68
female,Viral sinusitis (disorder),61
male,Viral sinusitis (disorder),51
female,Acute viral pharyngitis (disorder),38
male,Acute viral pharyngitis (disorder),28
female,Acute bronchitis (disorder),26
female,Miscarriage in first trimester,21
male,Acute bronchitis (disorder),20
male,Hypertension,19
male,Anemia (disorder),18


This query (`top_conditions_by_gender`) returns the 20 most common condition/gender combinations across the dataset, ranked by frequency.

## Stage 9: Extensions

With four Delta tables built from raw FHIR JSON entirely through manual upload and in-workspace transformation, several extensions were identified for future iterations:
 
- **Additional resource types** — Medication, Immunization, and Procedure follow the same explode-and-flatten pattern established here.
- **Incremental loads** — replacing `overwrite` with `MERGE INTO` to upsert new/changed records by ID, per standard Delta Lake practice.
- **Lightweight medallion separation** — maintaining a raw Delta table (minimally parsed) alongside a cleaned Delta table, mirroring medallion architecture concepts without additional infrastructure.
- **BI exploration** — connecting a SQL warehouse or Genie dashboard to the `observations` table.
- **Cross-project comparison** — comparing this pipeline's transformation approach against an equivalent AWS Glue/Redshift pipeline processing the same source data (DynamicFrames vs. plain PySpark).
